# Pyramidal HEALPix convolution -- single, small-region test

A simplified, fast companion to `pyramid_conv_sentinel2_test.ipynb`: instead
of running on the whole acquisition, this notebook crops a small lat/lon
square out of one real product and builds a **fresh, small**
`HealPixDecomp`/`HealPixKernelPyramid`/`HealPixPyramidConv` scoped to that
square only -- cheap enough to iterate on quickly, no risk of the OOM issues
a full-scene run at deep `Jmax` can hit.

Two tests, run through the **same** pyramid (built once, in section 3):

- **A. Real data** (section 4): the cropped RGB square, before/after.
- **B. Single Dirac at the square's centre** (section 5): with everything
  else zero, the response is -- by construction -- this pipeline's point
  -spread function (PSF). Horizontal and vertical cuts through the centre
  are plotted against the analytic `KERNEL_SHAPE` profile it was built
  from, as a direct visual/numeric consistency check: does what the
  pipeline actually does to a point match what it was asked to do?

**Not executed here** -- same reason as `pyramid_conv_sentinel2_test.ipynb`:
this sandbox has no network access to `data.grid4earth.eu`. Run it where
that notebook already runs (e.g. Datarmor). If `healpix_analyse` changed on
disk since the kernel started, **restart the kernel** before re-running
(see that notebook's own note on this -- it applies here identically).


## 1. Parameters

In [ ]:
import os, sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))          # to import g4e_source (next to this notebook)
sys.path.insert(0, str(Path.cwd().parent))    # to import healpix_analyse in dev mode

from g4e_source import G4E_L2A, G4E_PRODUCTS, RGB, ProductSeries

from healpix_analyse.decomp import HealPixDecomp
from healpix_analyse.kernel_pyramid import (
    HealPixKernelPyramid, kernel_gaussian, kernel_exponential, kernel_lorentzian, kernel_beta,
)
from healpix_analyse.pyramid_conv import HealPixPyramidConv
from healpix_analyse._ellipsoid import canonicalize_ellipsoid

# --------------------------------------------------------------------------
# Same convolution parameters as pyramid_conv_sentinel2_test.ipynb -- this
# notebook tests the *same* pipeline, just on a small crop instead of the
# whole scene.
# --------------------------------------------------------------------------
PRODUCTS    = list(G4E_PRODUCTS)
TIME_INDEX  = 0
DATA_LEVEL  = 17
SCALING     = "reflectance"
CACHE       = os.path.expanduser("~/s2_cache")

N_CHANNELS         = 3
JMAX               = 4     # fewer stages than the full-scene notebook (8): the
                            # crop below is small, a deep pyramid has nothing
                            # left to coarsen into past a few stages and just
                            # costs time -- raise it if SUB_N_CELLS (section 2)
                            # comes out large and you want to test a deeper PSF
COMPACT_KERNEL_SZ  = 5
KERNEL_SHAPE       = kernel_gaussian
SIGMA_PIX          = 1.2
GAUGE_TYPE         = "phi"

# How big a square to crop out of the scene, as a fraction of the scene's own
# half-width (same convention as pyramid_conv_sentinel2_test.ipynb section 8).
# Kept fairly generous here (vs. 0.15 there) on purpose: the Dirac test in
# section 5 needs enough margin around the centre that the pyramid's own
# reach at JMAX doesn't run into the crop's own edge before it decays --
# too tight a crop would show a boundary artefact (see section 8 of the
# full-scene notebook on domain-edge effects) instead of a clean PSF.
SUBTILE_FRACTION = 0.35

DTYPE  = torch.float32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


## 2. Reading a real acquisition and cropping a small square

Same read as `pyramid_conv_sentinel2_test.ipynb` section 2 (whole scene,
level 17 is already the coarsest published level, so this stays cheap), then
immediately restricted to a small lat/lon square around the scene's own
centre -- this crop is what sections 3-5 actually work on, not the full
scene.


In [ ]:
src = ProductSeries(PRODUCTS, level=DATA_LEVEL, base=G4E_L2A, bands=RGB, scaling=SCALING)
level, cell_id, dates = src.level, src.cell_id, src.dates
print(f"HEALPix level {level}, {cell_id.size} cells, product {dates[TIME_INDEX]}")

rgb = src.rgb(TIME_INDEX, cache=CACHE)
print(f"NaN in full scene: {100 * np.isnan(rgb).any(1).mean():.1f}% of cells")

import healpix_geo
lon_deg, lat_deg = healpix_geo.nested.healpix_to_lonlat(
    cell_id.tolist(), level, ellipsoid=canonicalize_ellipsoid(src.ellipsoid),
)
lon_deg, lat_deg = np.asarray(lon_deg), np.asarray(lat_deg)

lon0 = 0.5 * (lon_deg.min() + lon_deg.max())
lat0 = 0.5 * (lat_deg.min() + lat_deg.max())
half_width_deg = SUBTILE_FRACTION * 0.5 * max(
    lon_deg.max() - lon_deg.min(), lat_deg.max() - lat_deg.min(),
)
in_square = (np.abs(lon_deg - lon0) < half_width_deg) & (np.abs(lat_deg - lat0) < half_width_deg)

sub_cell_id = cell_id[in_square]
sub_lon = lon_deg[in_square]
sub_lat = lat_deg[in_square]
sub_rgb = rgb[in_square]
SUB_N_CELLS = sub_cell_id.size

print(f"square: {2 * half_width_deg:.3f}x{2 * half_width_deg:.3f} deg around "
      f"(lon={lon0:.3f}, lat={lat0:.3f}), {SUB_N_CELLS} cells "
      f"({100 * np.isnan(sub_rgb).any(1).mean():.1f}% NaN)")
if SUB_N_CELLS < 200:
    raise ValueError(
        "too few cells in this square -- raise SUBTILE_FRACTION, or this "
        "date/tile is mostly cloudy near its centre (try a different TIME_INDEX)"
    )

# nearest cell to the square's own centre -- this is where the Dirac in
# section 5 gets placed, and what every cross-section below is centred on
center_idx = int(np.argmin((sub_lon - lon0) ** 2 + (sub_lat - lat0) ** 2))
center_lon, center_lat = float(sub_lon[center_idx]), float(sub_lat[center_idx])
print(f"centre cell: index {center_idx} in the crop, at (lon={center_lon:.4f}, lat={center_lat:.4f})")


## 3. A fresh, small pyramid scoped to the crop

Built once, reused identically for both tests below (real data in section
4, Dirac in section 5) -- this is the actual point of the notebook: the
same pipeline is exercised on two different inputs so its response to a
point (section 5) can be read as a diagnostic for what it does to real data
(section 4), not as a separate, unrelated experiment.


In [ ]:
decomp = HealPixDecomp(
    level=level, cell_ids=sub_cell_id, Jmax=JMAX,
    ellipsoid=src.ellipsoid, dtype=DTYPE, device=DEVICE,
)
print(decomp)
print("band sizes (fine -> coarse):", decomp.sizes)

kernel_pyramid = HealPixKernelPyramid.from_kernel(
    decomp, KERNEL_SHAPE(sigma_pix=SIGMA_PIX),
    compact_kernel_sz=COMPACT_KERNEL_SZ, gauge_type=GAUGE_TYPE,
    channels=N_CHANNELS, ellipsoid=src.ellipsoid, dtype=DTYPE,
)
pconv = HealPixPyramidConv(decomp, kernel_pyramid, mode="normalized")

grid_plot = __import__("healpix_plot").HealpixGrid(
    level=level, indexing_scheme="nested", ellipsoid=canonicalize_ellipsoid(src.ellipsoid),
)
view = (sub_lon.min(), sub_lon.max(), sub_lat.min(), sub_lat.max())


## 4. Test A -- real cropped data, before / after

In [ ]:
import cartopy.crs as ccrs
import healpix_plot

x_real = torch.as_tensor(sub_rgb.T, dtype=DTYPE, device=DEVICE)
with torch.no_grad():
    y_real, support_real = pconv(x_real, return_support=True)
rgb_filtered = y_real.detach().cpu().numpy().T
support_map = support_real.detach().cpu().numpy()[0]

hi = float(np.nanpercentile(sub_rgb, 98))
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5),
                          subplot_kw={"projection": ccrs.PlateCarree()}, layout="constrained")
healpix_plot.plot(sub_cell_id, sub_rgb, healpix_grid=grid_plot, sampling_grid={"shape": 300},
                   view=view, ax=axes[0], rgb_clip=(0.0, hi), axis_labels="none", title="before (real)")
healpix_plot.plot(sub_cell_id, rgb_filtered, healpix_grid=grid_plot, sampling_grid={"shape": 300},
                   view=view, ax=axes[1], rgb_clip=(0.0, hi), axis_labels="none",
                   title=f"after (Jmax={JMAX}, {COMPACT_KERNEL_SZ}x{COMPACT_KERNEL_SZ})")
mp = healpix_plot.plot(sub_cell_id, support_map, healpix_grid=grid_plot, sampling_grid={"shape": 300},
                        view=view, ax=axes[2], axis_labels="none", title="synthesized support")
fig.colorbar(mp, ax=axes[2], shrink=0.7)
plt.show()


## 5. Test B -- single Dirac at the centre: the pipeline's own PSF

Everything zero except the centre cell (all 3 channels set to 1). The
output is, by construction, this exact pipeline's point-spread function --
what it actually does to a point, as opposed to what `KERNEL_SHAPE`/
`SIGMA_PIX` asked it to do. Plotted first as a 2D map, then as horizontal
and vertical cuts through the centre, overlaid on the analytic kernel
profile (both curves normalized to peak 1 -- this checks *shape*, not
absolute amplitude, since `mode="normalized"` rescales by the local
support, not by the kernel's own integral).


In [ ]:
rgb_dirac = np.zeros_like(sub_rgb, dtype=np.float32)
rgb_dirac[center_idx, :] = 1.0

x_dirac = torch.as_tensor(rgb_dirac.T, dtype=DTYPE, device=DEVICE)
with torch.no_grad():
    y_dirac, support_dirac = pconv(x_dirac, return_support=True)
rgb_psf = y_dirac.detach().cpu().numpy().T          # [SUB_N_CELLS, 3]
support_psf = support_dirac.detach().cpu().numpy()[0]

psf = rgb_psf[:, 0]   # the 3 channels are identical (block-diagonal, same profile -- see class docstring)
print(f"PSF: min {psf.min():.4g}, max {psf.max():.4g}, has NaN {np.isnan(psf).any()}")
print(f"support at centre: {support_psf[center_idx]:.4g} (domain max: {support_psf.max():.4g}) -- "
      "should be close to the domain max; if it is much lower, SUBTILE_FRACTION is too small "
      "for this JMAX (the crop's own edge is contaminating the centre, see section 1's note)")

fig, ax = plt.subplots(figsize=(5, 4.3), subplot_kw={"projection": ccrs.PlateCarree()}, layout="constrained")
mp = healpix_plot.plot(sub_cell_id, psf, healpix_grid=grid_plot, sampling_grid={"shape": 300},
                        view=view, ax=ax, axis_labels="none",
                        title=f"PSF (response to a centred Dirac, Jmax={JMAX})")
fig.colorbar(mp, ax=ax, shrink=0.8)
plt.show()


In [ ]:
# Resample the PSF onto a small regular lon/lat grid (nearest, same
# convention healpix_plot.plot uses internally) so horizontal/vertical cuts
# are simple array slices, then compare against the analytic profile.
from healpix_plot.resampling import resample as hp_resample

XSECT_SHAPE = 65   # odd, so there is an exact centre row/column
target_grid, psf_image = hp_resample(
    sub_cell_id, psf, sampling_grid={"shape": XSECT_SHAPE}, healpix_grid=grid_plot,
    interpolation="nearest", agg="mean",
)
xs = target_grid.x[0, :]   # longitudes, degrees
ys = target_grid.y[:, 0]   # latitudes, degrees
row_idx = int(np.argmin(np.abs(ys - center_lat)))   # horizontal cut: fixed latitude
col_idx = int(np.argmin(np.abs(xs - center_lon)))   # vertical cut:   fixed longitude


def haversine_deg(lon1, lat1, lon2, lat2):
    """Angular great-circle distance (radians) between (lon1,lat1) and (lon2,lat2), degrees in."""
    lon1r, lat1r, lon2r, lat2r = np.radians(lon1), np.radians(lat1), np.radians(lon2), np.radians(lat2)
    dlon, dlat = lon2r - lon1r, lat2r - lat1r
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon / 2) ** 2
    return 2 * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


# alpha_pix: the finest band's own angular pixel spacing, in radians -- the
# exact convention KERNEL_SHAPE(rho_pix, phi) is defined against (see the
# KernelFn docstring in healpix_analyse/kernel_pyramid.py).
nside = 2 ** level
alpha_pix = np.sqrt(4.0 * np.pi / (12.0 * nside ** 2))

h_row_lat = ys[row_idx]
theta_h = haversine_deg(xs, h_row_lat, center_lon, h_row_lat)
rho_h = np.sign(xs - center_lon) * theta_h / alpha_pix
cut_h = psf_image[row_idx, :]

v_col_lon = xs[col_idx]
theta_v = haversine_deg(v_col_lon, ys, v_col_lon, center_lat)
rho_v = np.sign(ys - center_lat) * theta_v / alpha_pix
cut_v = psf_image[:, col_idx]

kernel_fn = KERNEL_SHAPE(sigma_pix=SIGMA_PIX)
rho_theory = np.linspace(min(rho_h.min(), rho_v.min()), max(rho_h.max(), rho_v.max()), 400)
theory = kernel_fn(np.abs(rho_theory), np.zeros_like(rho_theory))

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for ax, rho, cut, label in (
    (axes[0], rho_h, cut_h, "horizontal cut (varying longitude, latitude fixed)"),
    (axes[1], rho_v, cut_v, "vertical cut (varying latitude, longitude fixed)"),
):
    finite = np.isfinite(cut)
    cut_peak = np.nanmax(np.abs(cut[finite])) if finite.any() else 1.0
    ax.plot(rho[finite], cut[finite] / max(cut_peak, 1e-12), "o-", ms=3, label="measured PSF (peak-normalized)")
    ax.plot(rho_theory, theory / theory.max(), "--", label=f"KERNEL_SHAPE (sigma_pix={SIGMA_PIX})")
    ax.axvline(0, color="0.7", lw=0.8)
    ax.set_xlabel("signed distance from centre (pixel units, finest band)")
    ax.set_title(label, fontsize=9)
    ax.legend(fontsize=8)
plt.show()


## Reading the result

- If the measured PSF (solid) tracks the dashed analytic curve closely near
  the centre, the pipeline reconstructs the requested kernel shape well at
  this `JMAX`/`SIGMA_PIX`/`COMPACT_KERNEL_SZ`.
- A measured curve **wider** than the analytic one, or with visible
  shoulders/ringing the analytic curve doesn't have, is exactly what the
  per-band, block-diagonal construction in `from_kernel` (as opposed to the
  jointly-fit `calibrate_joint`, see `docs/pyramid_convolution.md` section
  D.1) is expected to add on top of the requested profile once `Jmax > 0`
  -- each pyramid band reconstructs its own share of the response
  independently, so their sum is not guaranteed to reduce back to the
  single analytic kernel evaluated at the finest resolution. This section
  is exactly how to *see* that gap on a specific `SIGMA_PIX`/`JMAX`
  combination, rather than only reading about it in the docs.
- If the two curves disagree badly and `support at centre` printed above
  was also well below the domain max, re-run with a larger
  `SUBTILE_FRACTION` first (section 1) -- that disagreement would be a
  crop-boundary artefact, not a property of the kernel pyramid itself.
